In [1]:
!pip install -q pandas scikit-surprise


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/154.4 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!pip uninstall -y numpy
!pip install numpy==1.26.4


Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which i

In [1]:
!pip install scikit-surprise


In [29]:
import pandas as pd
import numpy as np
import os
from surprise import Dataset, SVD
from surprise.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler


In [34]:
data = Dataset.load_builtin('ml-100k', prompt=False)
trainset = data.build_full_trainset()

svd = SVD()
svd.fit(trainset)

print("Collaborative model trained!")


Collaborative model trained!


In [35]:
movies_path = "/root/.surprise_data/ml-100k/ml-100k/u.item"

movies = pd.read_csv(
    movies_path,
    sep="|",
    encoding="latin-1",
    header=None
)

genre_cols = movies.columns[5:]

movies = movies[[0,1] + list(genre_cols)]
movies.columns = ["movie_id", "title"] + list(genre_cols)

movies["movie_id"] = movies["movie_id"].astype(str)

print("Movie metadata loaded!")


Movie metadata loaded!


In [36]:
genre_matrix = movies.iloc[:, 2:].values
cosine_sim = cosine_similarity(genre_matrix)

print("Content similarity matrix built!")


Content similarity matrix built!


In [37]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

def hybrid_recommend(user_id, top_n=10):

    movie_ids = [trainset.to_raw_iid(i) for i in trainset.all_items()]

    collab_scores = []
    content_scores = []

    for idx, movie_id in enumerate(movie_ids):
        collab_scores.append(svd.predict(user_id, movie_id).est)
        content_scores.append(np.mean(cosine_sim[idx]))

    # Normalize both to 0–1 scale
    collab_scores = scaler.fit_transform(np.array(collab_scores).reshape(-1,1)).flatten()
    content_scores = scaler.fit_transform(np.array(content_scores).reshape(-1,1)).flatten()

    # Weighted combination
    hybrid_scores = alpha * collab_scores + (1 - alpha) * content_scores

    results = list(zip(movie_ids, hybrid_scores))
    results.sort(key=lambda x: x[1], reverse=True)

    top_movies = results[:top_n]

    top_df = pd.DataFrame(top_movies, columns=["movie_id", "hybrid_score"])
    top_df = top_df.merge(movies[["movie_id", "title"]], on="movie_id")

    return top_df[["title", "hybrid_score"]]


In [38]:
hybrid_recommend(user_id=196, top_n=10)


,title,hybrid_score
0,Shall We Dance? (1996),0.928447
1,"Wrong Trousers, The (1993)",0.926718
2,Rear Window (1954),0.924050
3,"Boot, Das (1981)",0.894596
4,Braveheart (1995),0.894072
5,Vertigo (1958),0.888347
6,Raging Bull (1980),0.886311
7,Dr. Strangelove or: How I Learned to Stop Worr...,0.882952
8,Good Will Hunting (1997),0.878659
9,Raise the Red Lantern (1991),0.876040


In [39]:
os.makedirs("models", exist_ok=True)

joblib.dump(svd, "models/svd_model.pkl")
joblib.dump(cosine_sim, "models/content_similarity.pkl")
joblib.dump(movies, "models/movies_metadata.pkl")
joblib.dump(alpha, "models/hybrid_alpha.pkl")

print("Hybrid model saved successfully!")


Hybrid model saved successfully!
